# 📘Домашнє завдання №12 Дерева рішень

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW12

## 📌 Завдання 1
Використайте датасет Titanic:  
https://www.kaggle.com/datasets/yasserh/titanic-dataset  

- Завантажте дані  
- Обробіть пропущені значення  
- Закодуйте категоріальні змінні (наприклад, `Sex`, `Embarked`)  

---

## 📌 Завдання 2
Побудуйте модель класифікації:

- Натренуйте `DecisionTreeClassifier`  
- Зробіть прогноз на тестових даних  
- Обчисліть метрики:
  - accuracy  
  - precision  
  - recall  

👉 Проаналізуйте отримані результати

---

## 📌 Завдання 3
Візуалізуйте дерево рішень:

- Виведіть структуру дерева (`plot_tree` або аналог)  
- Проаналізуйте:
  - які ознаки використовуються найчастіше  
  - які правила формуються  

---

## 📌 Завдання 4
Використайте датасет Fish Market:  
https://www.kaggle.com/datasets/vipullrathod/fish-market  

- Побудуйте модель `DecisionTreeRegressor`  
- Натренуйте модель для прогнозування (наприклад, `Weight`)  
- Обчисліть метрики:
  - MSE  
  - R²  

---

## 📌 Завдання 5
Оптимізація моделі:

- Використайте `GridSearchCV` для підбору гіперпараметрів (наприклад: `max_depth`, `min_samples_leaf`)  
- Знайдіть найкращу модель  
- Порівняйте метрики з результатами з **Завдання 4**  

👉 Зробіть висновок:
- чи покращилась модель після підбору параметрів?

In [1]:
# Synchronization with remote source

import shutil
from pathlib import Path

# Input data
git_project_url = "https://github.com/BogdanPinchuk/DataScience-PBY_HW12.git"
main_file_name = "Bohdan_Pinchuk_DS_HW12.ipynb"

# Solution

# upload all files
current_path = !pwd
current_path = current_path[0]
parent_path = !dirname "$current_path"
parent_path = parent_path[0]
temp_path = f"{parent_path}/temp"

# Clone data
!rm -rf "$temp_path"
!git clone "$git_project_url" "$temp_path"

source = Path(temp_path)
destination = Path(current_path)
exclude = {main_file_name, ".git", ".idea"}

for item in source.iterdir():
    if item.name in exclude:
        continue

    target = destination / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# Clean temp folder
!rm -rf "$temp_path"

Cloning into '/Users/bohdanpinchuk/Documents/Data Science/Development/Data_Science/Practical_tasks/Homework_12/temp'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 29 (delta 4), reused 28 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 46.08 KiB | 773.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.


In [2]:
# Download data (to cover the case when the data aren't accessible)

# !pip install kagglehub[pandas-datasets]
# !pip install ipywidgets
# !pip install --upgrade nbformat
# !pip install jinja2

import sqlite3
import kagglehub
import pandas as pd
from pandas import DataFrame
from kagglehub import KaggleDatasetAdapter

# Input data
db_file_name = "store_hw12.db"


# Solution
def download_and_extract_from_kagglehub(ds_path: str, ds_file_name: str) -> DataFrame | None:
    """
    Download and extract data from kagglehub
    :param ds_path: path to kaggle dataset
    :param ds_file_name: name of kaggle dataset
    :return: DataFrame or None
    """
    ds_name = ds_path.split("/")[-1].replace('-', '_')

    # Note: to handle error: "SSL: CERTIFICATE_VERIFY_FAILED" or no connection to the server
    try:
        # for testing
        # raise Exception

        ds_data = kagglehub.dataset_load(
            KaggleDatasetAdapter.PANDAS,
            ds_path,
            ds_file_name,
        )

        file_path = Path(db_file_name)

        # Use only one time to initialize/update data (at first time)
        if not file_path.exists():
            conn = sqlite3.connect(db_file_name)
            ds_data.to_sql(ds_name, conn, if_exists="replace", index=False)
            conn.close()
    except Exception:
        conn = sqlite3.connect(db_file_name)
        ds_data = pd.read_sql(f"SELECT * FROM {ds_name}", conn)
        conn.close()

    return ds_data


In [3]:
## Отримання даних

# Solution
titanic_dataset = download_and_extract_from_kagglehub("yasserh/titanic-dataset", "Titanic-Dataset.csv")
fish_market = download_and_extract_from_kagglehub("vipullrathod/fish-market", "Fish.csv")

# Print results
display(titanic_dataset, fish_market)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340
...,...,...,...,...,...,...,...
154,Smelt,12.2,11.5,12.2,13.4,2.0904,1.3936
155,Smelt,13.4,11.7,12.4,13.5,2.4300,1.2690
156,Smelt,12.2,12.1,13.0,13.8,2.2770,1.2558
157,Smelt,19.7,13.2,14.3,15.2,2.8728,2.0672
